# py-sigxtalk Quickstart: Full Analysis & Visualization

This notebook demonstrates the complete SigXTalk analysis pipeline in Python using the PBMC3k dataset,
including all 11 visualization functions. All figures are saved to `figures/`.

In [ ]:
import sys, os
sys.path.insert(0, '../src')
import pysigxtalk as psx
import anndata
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)
print(f"py-sigxtalk version: {psx.__version__}")
print(f"Figures will be saved to: {FIG_DIR}/")

## 1. Load PBMC3k Data

In [ ]:
adata = anndata.read_h5ad('pbmc3k_final.h5ad')
print(f"Shape: {adata.shape}")
print(f"Cell types: {adata.obs['cell_type'].value_counts().to_dict()}")

## 2. Load Databases

In [ ]:
rtf_db, tftg_db = psx.load_databases(species="human")
print(f"RTF: {len(rtf_db):,}, TFTG: {len(tftg_db):,}")

## 3. Extract Target Cell Type & Prepare Data

In [ ]:
target_type = 'CD14+ Mono'
target_mask = adata.obs['cell_type'] == target_type
adata_target = adata[target_mask].copy()

X = adata_target.layers['scale_data'] if 'scale_data' in adata_target.layers else adata_target.X
exp_mat = pd.DataFrame(
    X.T.toarray() if hasattr(X, 'toarray') else X.T,
    index=adata_target.var.index.tolist(),
    columns=adata_target.obs.index.tolist()
)
exp_mat = psx.get_exp_clu(exp_mat, cutoff=0.1)
print(f"Expression: {exp_mat.shape}")

target_genes = exp_mat.var(axis=1).nlargest(200).index.tolist()
print(f"Target genes: {len(target_genes)}")

all_genes = exp_mat.index.tolist()
receptors = [g for g in rtf_db['from'].unique() if g in all_genes][:20]
np.random.seed(42)
lr_pairs = pd.DataFrame({
    'Ligand': np.random.choice(receptors, 30, replace=True),
    'Receptor': np.random.choice(receptors, 30, replace=True),
    'Weight': np.random.rand(30) * 0.8 + 0.2,
})
print(f"LR pairs: {len(lr_pairs)}")

## 4. Prepare HGNN Inputs

In [ ]:
inputs = psx.prepare_input(
    exp_mat=exp_mat, target_genes=target_genes,
    lr_pairs=lr_pairs, rtf_db=rtf_db, tftg_db=tftg_db,
)
print(f"Expression: {inputs.exp_clu.shape}")
print(f"RTF: {len(inputs.rtf_filtered)}, TFTG: {len(inputs.tftg_filtered)}")

## 5. Run HGNN Model

In [ ]:
pathways = psx.run_hgnn(
    inputs, epochs=30, device="cpu", seed=42,
    hgnn_dims=[64, 32], linear_dims=[16, 8]
)
print(f"Pathways: {len(pathways)}")
print(f"Active (pred > 0.5): {len(pathways[pathways['pred_label'] > 0.5])}")

## 6. Calculate PRS

In [ ]:
prs = psx.compute_prs(inputs.exp_clu, pathways, engine="sklearn", n_estimators=100, cutoff=0.5)
prs_filtered = psx.filter_results(prs, prs_threshold=0.01)
print(f"PRS results: {len(prs_filtered)} pathways")

## 7. Crosstalk Analysis

In [ ]:
if len(prs_filtered) > 0:
    counts = psx.count_crosstalk(prs_filtered, data_type="Target")
    trs = psx.aggregate_causality(prs_filtered, data_type="Target")
    top_target = counts.idxmax()
    print(f"Crosstalk: {len(counts)} genes, {len(trs)} TRS pairs")
    print(f"Top target: {top_target} ({counts.max()} pathways)")
else:
    print("No PRS results, using synthetic data for demo")
    genes = exp_mat.index[:30].tolist()
    prs_filtered = pd.DataFrame({
        'Receptor': np.random.choice(genes[:10], 100),
        'SSC': np.random.choice(genes[10:20], 100),
        'Target': np.random.choice(genes[20:], 100),
        'Weight': np.random.rand(100),
    })
    counts = psx.count_crosstalk(prs_filtered, data_type="Target")
    trs = psx.aggregate_causality(prs_filtered, data_type="Target")
    top_target = counts.idxmax()

## 8. Visualization 1: Crosstalk Counts (Histogram)

In [ ]:
fig = psx.plot_counts_histogram(prs_filtered, data_type="Target")
fig.savefig(f"{FIG_DIR}/01a_crosstalk_histogram.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {FIG_DIR}/01a_crosstalk_histogram.png")

fig = psx.plot_counts_bar(prs_filtered, data_type="Target", top_percent=10)
fig.savefig(f"{FIG_DIR}/01b_crosstalk_bar.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {FIG_DIR}/01b_crosstalk_bar.png")

## 9. Visualization 2: Heatmap

In [ ]:
fig = psx.plot_heatmap(prs_filtered, gene_used=top_target, genetype="Target")
fig.savefig(f"{FIG_DIR}/02_heatmap.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {FIG_DIR}/02_heatmap.png")

## 10. Visualization 3: Fidelity & Specificity

In [ ]:
fig = psx.plot_fid_spe(prs_filtered, key_tg=top_target, threshold=0.0)
fig.savefig(f"{FIG_DIR}/03_fid_spe.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {FIG_DIR}/03_fid_spe.png")

## 11. Visualization 4: Alluvial Diagram

In [ ]:
fig = psx.plot_alluvial(prs_filtered, key_tg=top_target, min_weight=0.0)
fig.savefig(f"{FIG_DIR}/04_alluvial.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {FIG_DIR}/04_alluvial.png")

## 12. Visualization 5: Ridgeline Plot

In [ ]:
fig = psx.plot_ridgeline(prs_filtered, key_tg=top_target)
fig.savefig(f"{FIG_DIR}/05_ridgeline.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {FIG_DIR}/05_ridgeline.png")

## 13. Visualization 6: Chord Diagram

In [ ]:
fid_matrix = psx.calculate_fidelity_matrix(prs_filtered, key_tg=top_target, mode="SSC")
if not fid_matrix.empty:
    fig = psx.plot_chord(fid_matrix)
    fig.savefig(f"{FIG_DIR}/06_chord.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {FIG_DIR}/06_chord.png")
else:
    print("No fidelity data for chord diagram")

## 14. Visualization 7: CCI Chord

In [ ]:
# Create synthetic CCI results for demo
cci_results = pd.DataFrame({
    'Ligand': np.random.choice(receptors[:10], 30),
    'Receptor': np.random.choice(receptors[:15], 30),
    'Weight': np.random.rand(30),
    'Source': np.random.choice(['T', 'Mono', 'B', 'NK'], 30),
    'Target': [target_type]*30,
})
fig = psx.plot_cci_chord(cci_results, topk=10)
fig.savefig(f"{FIG_DIR}/07_cci_chord.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {FIG_DIR}/07_cci_chord.png")

## 15. Visualization 8: CCI Circle

In [ ]:
fig = psx.plot_cci_circle(cci_results, topk=10)
fig.savefig(f"{FIG_DIR}/08_cci_circle.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {FIG_DIR}/08_cci_circle.png")

## 16. Visualization 9: Signal Contribution

In [ ]:
fig = psx.plot_signal_contribution(trs, inputs.exp_clu, key_tg=top_target)
fig.savefig(f"{FIG_DIR}/09_signal_contribution.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {FIG_DIR}/09_signal_contribution.png")

## 17. Visualization 10: Rec-TG Heatmap

In [ ]:
fig = psx.plot_rec_tg_heatmap(trs, inputs.exp_clu, key_tg=top_target)
fig.savefig(f"{FIG_DIR}/10_rec_tg_heatmap.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {FIG_DIR}/10_rec_tg_heatmap.png")

## 18. Visualization 11: Circular Bar Chart

In [ ]:
fig = psx.plot_circular_bar(prs_filtered, topk=5)
fig.savefig(f"{FIG_DIR}/11_circular_bar.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {FIG_DIR}/11_circular_bar.png")

## 19. Save Results

In [ ]:
prs_filtered.to_csv("prs_results.csv", index=False)
trs.to_csv("trs_results.csv", index=False)
pathways.to_csv("pathways.csv", index=False)

print("Results saved!")
print(f"  prs_results.csv: {len(prs_filtered)} rows")
print(f"  trs_results.csv: {len(trs)} rows")
print(f"  pathways.csv: {len(pathways)} rows")

print(f"\nFigures saved to {FIG_DIR}/:")
for f in sorted(os.listdir(FIG_DIR)):
    size = os.path.getsize(os.path.join(FIG_DIR, f))
    print(f"  {f}: {size/1024:.1f} KB")